<a href="https://colab.research.google.com/github/MariiaYarmolenko/HW-Data-Loves/blob/main/HW_11.4_%D0%86%D0%BD%D1%82%D0%B5%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%B2%D0%BD%D1%96_%D0%B2%D1%96%D0%B7%D1%83%D0%B0%D0%BB%D1%96%D0%B7%D0%B0%D1%86%D1%96%D1%97_%D0%B7_Plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Інтерактивні візуалізації з Plotly

## Опис завдання
У цьому домашньому завданні ви будете створювати інтерактивні візуалізації з допомогою бібліотеки Plotly. Ви дізнаєтесь різницю між Plotly Express (швидкі графіки) та Graph Objects (повний контроль), та створите інтерактивний дашборд.

**Опис колонок:**
- `datetime` - дата та час
- `season` - квартал (1-Q1, 2-Q2, 3-Q3, 4-Q4)
- `holiday` - чи є день святковим (0=ні, 1=так)
- `workingday` - чи є день робочим (0=ні, 1=так)
- `weather` - погодні умови (1=ясно, 2=туман, 3=легкий дощ, 4=сильний дощ)
- `temp` - температура в градусах Цельсія
- `atemp` - відчувається як температура
- `humidity` - вологість (%)
- `windspeed` - швидкість вітру
- `casual` - кількість випадкових користувачів
- `registered` - кількість зареєстрованих користувачів
- `count` - загальна кількість орендованих велосипедів

## Підготовка даних

---

🌱 Коментар щодо сезонності

Колонка season у датасеті представляє саме квартали року, а не метеорологічні сезони. Тому всі аналізи сезонності ви можете будувати на основі кварталів.

Водночас дані були зібрані в Індії, де поділ на сезони інший, ніж у Європі чи США. Якщо ви хочете дослідити сезонність відповідно до індійської системи сезонів, можна створити окрему колонку.

Справжні сезони в Індії:

| Сезон        | Місяці                     |
| ------------ | -------------------------- |
| Winter       | December–February (12,1,2) |
| Summer       | March–May (3,4,5)          |
| Monsoon      | June–September (6,7,8,9)   |
| Post-monsoon | October–November (10,11)   |


Тоді потрібно зробити нову колонку weather_season_india, мапнувши місяці так:

12, 1, 2 → 1 (Winter)

3, 4, 5 → 2 (Summer)

6–9 → 3 (Monsoon)

10–11 → 4 (Post-Monsoon)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Завантаження даних
df = pd.read_csv('/content/drive/MyDrive/Data/yulu_rental.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# Для plotly краще не встановлювати datetime як індекс
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.day_name()

# Додаємо назви кварталів в окрему колонку
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
df['quarter_name'] = df['season'].map(quarter_map)

In [5]:
pip install -U kaleido

## Завдання 1: Базовий інтерактивний лінійний графік (Plotly Express)

**Завдання:**
Створіть інтерактивний лінійний графік динаміки оренди за часом (рівень деталізації - як в даних) з можливістю zoom та hover.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яка перевага інтерактивного графіка над статичним?
2. Чому на графіку є "пробіли" - ділянки, де одна пряма лінія зʼєднує два "суцільних" блоки з даними? Як би ви це могли дослідити на статичному графіку?


In [6]:
fig_line = px.line(
    df,
    x='datetime',
    y='count',
    title='Інтерактивний графік оренди',
    labels={'count': 'Кількість оренд', 'datetime': 'Дата та час'})

fig_line.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label='1 міс', step='month', stepmode='backward'),
            dict(count=6, label='6 міс', step='month', stepmode='backward'),
            dict(step='all')
        ])
    )
)

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




1. Перевага інтерактивного графіка над статичним полягає у можливості деталізації (Zooming), тобто можна роздивитись конкретний тиждень чи день, щоб побачити дрібні коливання, які на статичному графіку злилися б в одну лінію.

Точність (Hover), тобто при наведенні курсору на точку ми отримуємо конкретні значення count і datetime, тоді як на статичному графіку нам довелося б "на око" визначати значення по осях.

2. Прямі лінії виникають, тому що бібліотека будує лінію від останнього наявного запису до першого запису в наступному блоці даних. Якщо між ними є декілька годин або днів пропущених даних, то лінія просто з'єднує ці точки по прямій.

На статичному графіку ми також бачили ці лінії, але проаналізувати його складніше. Можна побудувати графік, де по осі $x$ буде не дата, а послідовний індекс запису. Якщо лінії зникають, значить, проблема була в часових проміжках.
Можна використати код df.isnull().sum() або df.resample('H').count(), щоб знайти, де саме в даних "дірки".

## Завдання 2: Scatter plot з додатковими даними (Plotly Express)

**Завдання:**
Створіть scatter plot кількості орендованих велосипедів випадковими користувачами vs кількості орендованих велосипедів зареєстрованими користувачами. Розмір точок встановіть за сумарною кількістю велосипедів, які були взяті в оренду, а колір - за сезоном(кварталом). В hover_data - додайте деталі, які допоможуть вам в подальшому аналізі.

Дослідіть графік. Зверніть увагу, що ви можете вмикати і вимикати окремі квартали, якщо будете клікати на колір кварталу в легенді графіку.

**Дайте відповідь на питання.**
- Як ви проінтерпретуєте роздвоєність цього графіку (дві явні лінії)? Що це означає?
- Які висновки для компанії, які дає велосипеди в оренду, ви можете зробити з цього графіку? 3 основних висновки.

In [7]:
fig_scat = px.scatter(df.sample(1000),
                 x='casual',
                 y='registered',
                 size='count',
                 color='quarter_name',
                 hover_data=['temp', 'humidity', 'workingday', 'holiday'],
                 title='Аналіз поведінки випадкових та зареєстрованих користувачів',
                 labels={'registered': 'Зареєстровані', 'casual': 'Випадкові'}
                 )

fig_scat.show()

1. Велика концентрація точок ліворуч та вгору - це дні, коли кількість випадкових користувачів відносно невисока, а зареєстрованих - помітно більша. Це типові будні або прохолодні дні, коли попит формується переважно регулярними користувачами, які використовують велосипед для поїздок на роботу чи навчання.

Група точок праворуч та внизу - це дні з високим попитом, де кількість обох типів користувачів зростає. Це вихідні/ святкові дні (або просто теплі дні), коли сервіс стає інструментом дозвілля.

2. 3 основні висновки для компанії:
А) Зареєстровані користувачі забезпечують стабільність бізнесу, тому для них важливо підтримувати доступність парковок та справність велосипедів, особливо у години пік.
Випадкові користувачі - це джерело додаткового прибутку у вихідні та святкові дні, тому для них варто розробляти спеціальні «прогулянкові» тарифи та маркетингові акції під погоду.

Б) Велика кількість точок у верхньому правому секторі графіка свідчить, що саме в дні, коли випадкові користувачі стають активними, загальний дохід (розмір точок) суттєво зростає. Компанії варто інвестувати в прогнозування попиту на основі метеоданих, щоб оптимізувати кількість велосипедів у конкретних локаціях.

В) Наявність чіткого кластера зареєстрованих користувачів, які користуються сервісом навіть тоді, коли активність випадкових користувачів близька до нуля, вказує на успішну модель підписок або програми лояльності.

## Завдання 3: Порівняння Plotly Express vs Graph Objects

**Завдання:**
Створіть лінійний графік помісячної динаміки оренди (кількість оренд) велосипедів двома способами - з Plotly Express та з Graph Objects.

**Дайте відповіді на питання.**
1. Як ви розумієте основну різницю між цими двома підходами?
2. Коли краще використовувати Plotly Express?
3. Коли потрібен Graph Objects?


In [8]:
df['datetime'] = pd.to_datetime(df['datetime'])
df_monthly = df.set_index('datetime').resample('ME')['count'].sum().reset_index()

fig_px = px.line(
    df_monthly,
    x='datetime',
    y='count',
    title='Помісячна оренда',
    labels={'datetime': 'Місяць', 'count': 'Кількість оренд'}
)
fig_px.show()

In [9]:
fig_go = go.Figure()

fig_go.add_trace(go.Scatter(
    x=df_monthly['datetime'],
    y=df_monthly['count'],
    name='Оренди'
))

fig_go.update_layout(
    title='Помісячна оренда',
    xaxis_title='Місяць',
    yaxis_title='Кількість оренд'
)
fig_go.show()

In [10]:
fig_px_go = make_subplots(rows=1, cols=2, subplot_titles=('Plotly Express', 'Graph Objects'))

fig_px = px.line(df_monthly, x='datetime', y='count')

for trace in fig_px.data:
    fig_px_go.add_trace(trace, row=1, col=1)

fig_px_go.add_trace(go.Scatter(
    x=df_monthly['datetime'],
    y=df_monthly['count'],
    name='Graph Objects'
), row=1, col=2)

fig_px_go.update_layout(height=500, width=1300, title_text="Помісячна оренда")
fig_px_go.show()

1. Plotly Express фокусується на тому, що ми хочемо отримати і автоматично бере на себе налаштування осей, легенд, кольорових палітр та шаблонів.

Graph Objects фокусується на тому, як саме кожен елемент фігури має виглядати. Ми вручну визначаємо кожен "слід" (trace), налаштовуємо кожен маркер, лінію та властивість осей.

2. PX краще використовувати, коли:

- Потрібен швидкий EDA (Exploratory Data Analysis).

- Потрібен звичайний Scatter, Line, Bar, Box або Histogram.

- Потрібно швидко використати параметри color, size, facet_row, facet_col, щоб швидко розділити дані за категоріями.

3. GO, якщо:

- Потрібно змінити вигляд конкретного елемента, який PX не дозволяє змінити (наприклад, складні комбінації різних типів графіків на одній панелі).

- Потрібно створити рідкісні типи графіків (3D-поверхні, складні фінансові "свічки" або діаграми Санкі), які не підтримуються в PX.

- На етапі створення фінальниого інтерфейсу продукту, де кожна деталь (відступ, колір при наведенні, шрифт) повинна суворо відповідати бренду компанії.

- Якщо графік повинен змінюватися в реальному часі через складні callbacks (наприклад, у Dash).

## Завдання 4 (Опціональне): Дашборд з make_subplots (Graph Objects)

**Завдання:**
Створіть дашборд з 4 різними графіками в одній фігурі:
- Bar chart - середні значення загальної кількості оренд велосипедів за сезонами(кварталами)
- Pie chart - відсоткове співвідношення погодних умов в даних
- Line chart - середнє значення загальної кількості оренд велосипедів за годинами протягом доби
- Scatter plot - кореляція температури vs вологість

Додайте заголовок на дашборд.

**Дайте відповідь на питання**
- На ваш погляд, яка перевага об'єднання графіків в один дашборд?

In [11]:
season_avg = df.groupby('quarter_name')['count'].mean().reset_index()
weather_pie = df['weather'].value_counts().reset_index()
hour_avg = df.groupby('hour')['count'].mean().reset_index()

fig_subpl = make_subplots(
    rows=2, cols=2,
    specs = [[{"type": "xy"}, {"type": "domain"}],
         [{"type": "xy"}, {"type": "xy"}]],
    subplot_titles=('Середня оренда за сезонами', 'Розподіл погодних умов',
                    'Середня оренда за годинами', 'Температура vs Вологість')
)

fig_subpl.add_trace(go.Bar(x=season_avg['quarter_name'], y=season_avg['count'], name='Сезони'), row=1, col=1)

fig_subpl.add_trace(go.Pie(labels=weather_pie['weather'], values=weather_pie['count'], name='Погода'), row=1, col=2)

fig_subpl.add_trace(go.Scatter(x=hour_avg['hour'], y=hour_avg['count'], mode='lines', name='Години'), row=2, col=1)

fig_subpl.add_trace(go.Scatter(x=df['temp'], y=df['humidity'], mode='markers', name='Температура/Вологість'), row=2, col=2)

fig_subpl.update_layout(height=800, width=1000, title={
        'text': "Аналітичний дашборд оренди велосипедів",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    })
fig_subpl.show()

Перевага об'єднання графіків на один дашборд полягає у швидкості аналізу та формування цілісної картини.

## Завдання 5 (Опціональне): 3D візуалізація

**Завдання:**
Створіть 3D scatter plot для аналізу взаємозв'язку температури, швидкості вітру та загальної кількості орендованих велосипедів. Колір встановіть за сезоном(кварталом), а розмір - за загальною кількість оренд також.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яку додаткову інформацію, на ваш погляд, дає 3D візуалізація?
2. Чи видно кластери в 3D просторі?
3. Чи ви можете зробити висновки з цієї візуалізації, чи вам було простіше побудувати кілька 2D?



In [12]:
fig_3d = go.Figure(data=[go.Scatter3d(
    x=df['temp'],
    y=df['windspeed'],
    z=df['count'],
    mode='markers',
    marker=dict(
        size=df['count'] / 50,
        color=df['season'],
        opacity=0.6,
        colorscale='Viridis'
    )
)])

fig_3d.update_layout(
    title={
        'text': "3D аналіз: Температура vs Вітер vs Кількість оренд",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    scene=dict(
        xaxis_title='Температура',
        yaxis_title='Швидкість вітру',
        zaxis_title='Кількість оренд'
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

fig_3d.show()


## Завдання 6: Експорт та збереження інтерактивних графіків

**Завдання:**
Збережіть побудований попередній графік на plotly в формат HTML. Також змініть вручну щось на графіку (зум, виділення частини графіку) і збережіть його як статичне зображення через іконку фотоапарату у формат PNG. Завантажте файли з графіком у HTML та PNG (або посилання на них на github) разом з посиланням на цей ноутбук при здачі ДЗ.


In [13]:
fig_px.write_html("HW_dashboard.html")